# TeachAgent Diagnosis + Coach Full Demo

这个 notebook 用来完整测试：

- 环境检查
- `DiagnosisAgent` 诊断错因
- `CoachAgent` 接手教学
- 每一轮的 `quality / mode / source / stop_reason`
- session 状态变化


In [25]:
%pip install -U azure-ai-projects azure-identity openai nest_asyncio pydantic


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /opt/anaconda3/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
import sys
import importlib
import json

sys.path.append('/Users/xumuchi/Desktop/TeachAgent')

import coach_agent
import diagnosis_agent
importlib.reload(coach_agent)
importlib.reload(diagnosis_agent)

print('=== Coach Environment ===')
print(json.dumps(coach_agent.diagnose_environment(), ensure_ascii=False, indent=2))
print('=== Diagnosis Environment ===')
print(json.dumps(diagnosis_agent.diagnosis_environment(), ensure_ascii=False, indent=2))

=== Coach Environment ===
{
  "coach_agent_version": "2026-06-16-structured-output-check",
  "az_path": "/opt/homebrew/bin/az",
  "project_endpoint": "https://teachagent-xumuchi-20260614.services.ai.azure.com/api/projects/proj-default",
  "model_deployment": "gpt-4o-mini",
  "structured_outputs_supported": "True"
}
=== Diagnosis Environment ===
{
  "diagnosis_agent_version": "2026-06-17-fixed-problem-diagnosis",
  "az_path": "/Users/xumuchi/.local/bin:/Users/xumuchi/.pyenv/shims:/opt/homebrew/bin:/opt/anaconda3/bin:/opt/anaconda3/condabin:/Library/Frameworks/Python.framework/Versions/3.10/bin:/opt/homebrew/bin:/opt/homebrew/sbin:/Library/Frameworks/Python.framework/Versions/3.14/bin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/Library/

In [27]:
diag_agent = diagnosis_agent.FoundryDiagnosisAgent()
diag_session = diag_agent.create_session(
    problem_text='现有 7 名同学：甲、乙、丙、丁、戊、己、庚，先将 7 人分成一组 3 人、两组 2 人的三个非空小组，再将三个小组派往 A、B、C 三个不同社区做志愿。要求：①甲、乙不能同组；②若丙、丁分到同一组，则该小组不能派往 A 社区。求一共有多少种不同分配方案？',
    reference_answer='先算出无限制总分配方案 630 种，减去甲乙同组的违规方案 150 种，再减去甲乙不同组但丙丁同组且该组去 A 社区的违规方案 42 种，630-150-42=438，最终共 438 种。',
    student_profile='学生对这种题目没有任何思路',
    coach_max_turns=4,
)

student_answer = '我知道公式，但是我不知道从哪着手'
diagnosis = diag_agent.diagnose(student_answer, session=diag_session)

print('=== Diagnosis Result ===')
print(json.dumps(diagnosis.as_dict(), ensure_ascii=False, indent=2))
print('=== Diagnosis Session Snapshot ===')
print(json.dumps({
    'ocr_source': diag_session.ocr_source,
    'coach_max_turns': diag_session.coach_max_turns,
    'messages_len': len(diag_session.messages),
}, ensure_ascii=False, indent=2))

=== Diagnosis Result ===
{
  "problem_text": "现有 7 名同学：甲、乙、丙、丁、戊、己、庚，先将 7 人分成一组 3 人、两组 2 人的三个非空小组，再将三个小组派往 A、B、C 三个不同社区做志愿。要求：①甲、乙不能同组；②若丙、丁分到同一组，则该小组不能派往 A 社区。求一共有多少种不同分配方案？",
  "reference_answer": "先算出无限制总分配方案 630 种，减去甲乙同组的违规方案 150 种，再减去甲乙不同组但丙丁同组且该组去 A 社区的违规方案 42 种，630-150-42=438，最终共 438 种。",
  "student_answer": "我知道公式，但是我不知道从哪着手",
  "student_profile": "学生对这种题目没有任何思路",
  "error_type": "missing_strategy",
  "confidence": 0.35,
  "reason": "AI 诊断调用失败，已使用本地 fallback：TypeError",
  "evidence": "学生回答为空或直接表示不会。",
  "coach_mode": "socratic_light",
  "coach_trap": "学生知道局部知识，但不会先确定中间目标。",
  "coach_prompt": "学生完全没思路。先给一个很小的起点，只透露下一步，不要直接给完整答案。",
  "source": "fallback_heuristic"
}
=== Diagnosis Session Snapshot ===
{
  "ocr_source": "fixed_text",
  "coach_max_turns": 4,
  "messages_len": 3
}


In [28]:
coach = coach_agent.FoundryCoachAgent()
coach_session = diagnosis.build_coach_session(coach, max_turns=4)

print('=== Coach Session Snapshot ===')
print(json.dumps({
    'problem_text': coach_session.problem_text,
    'error_type': coach_session.error_type.value,
    'turn_index': coach_session.turn_index,
    'max_turns': coach_session.max_turns,
    'student_profile': coach_session.student_profile,
    'messages_len': len(coach_session.messages),
}, ensure_ascii=False, indent=2))

=== Coach Session Snapshot ===
{
  "problem_text": "现有 7 名同学：甲、乙、丙、丁、戊、己、庚，先将 7 人分成一组 3 人、两组 2 人的三个非空小组，再将三个小组派往 A、B、C 三个不同社区做志愿。要求：①甲、乙不能同组；②若丙、丁分到同一组，则该小组不能派往 A 社区。求一共有多少种不同分配方案？",
  "error_type": "missing_strategy",
  "turn_index": 0,
  "max_turns": 4,
  "student_profile": "学生对这种题目没有任何思路",
  "messages_len": 1
}


In [29]:
def dump_response(label, result, session, student_reply):
    print(f'=== {label} ===')
    print('student_reply:', student_reply)
    print('coach_content:', result.content)
    print('reply_quality:', result.reply_quality.value)
    print('analysis_source:', result.reply_analysis.source)
    print('analysis_reason:', result.reply_analysis.reason)
    print('mode:', result.strategy.mode.value)
    print('trap:', result.strategy.trap)
    print('strategy_prompt:', result.strategy.prompt)
    print('turn_index:', result.turn_index)
    print('done:', result.done)
    print('stop_reason:', result.stop_reason)
    print('session_messages_len:', len(session.messages))
    print('last_message_role:', session.messages[-1]['role'] if session.messages else None)
    print('last_two_messages:')
    print(json.dumps(session.messages[-2:], ensure_ascii=False, indent=2))
    print('-' * 60)

In [30]:
student_replies = [
    '是不是先算出所有无限制的分配总数？',
    '再分别减去两种不符合要求的违规情况？',
    '所以 630-150-42=438 种',
    '那我以后多限制排列组合先算全集再剔除违规。',
]

for i, reply in enumerate(student_replies, 1):
    result = coach.reply(reply, session=coach_session)
    dump_response(f'Turn {i}', result, coach_session, reply)

=== Turn 1 ===
student_reply: 是不是先算出所有无限制的分配总数？
coach_content: 对的，先算出在没有任何限制时把 7 个人分成 3‑2‑2 的三组并派往 A、B、C 的所有可能数是个好起点。接下来，你可以思考：在这个总数的基础上，怎样把“甲、乙不能同组”和“若丙、丁同组则不能去 A 社区”这两个条件逐个剔除不符合的情况呢？你准备先处理哪一个条件？
reply_quality: good
analysis_source: fallback_heuristic
analysis_reason: 模型输出不是可解析的 JSON，已使用本地 fallback。
mode: socratic_light
trap: 学生知道局部知识，但不会先确定中间目标。
strategy_prompt: 学生方向基本正确，但当前题还没答完整。先肯定，再只追问当前题缺的那一步，不要给变式。
turn_index: 1
done: False
stop_reason: continue
session_messages_len: 3
last_message_role: assistant
last_two_messages:
[
  {
    "role": "user",
    "content": "【题目】\n现有 7 名同学：甲、乙、丙、丁、戊、己、庚，先将 7 人分成一组 3 人、两组 2 人的三个非空小组，再将三个小组派往 A、B、C 三个不同社区做志愿。要求：①甲、乙不能同组；②若丙、丁分到同一组，则该小组不能派往 A 社区。求一共有多少种不同分配方案？\n\n【学生画像】\n学生对这种题目没有任何思路\n\n【学生错误类型】\nmissing_strategy\n\n【学生最新回答】\n是不是先算出所有无限制的分配总数？\n\n【学生理解度分析工具结果】\n回答质量：good\n是否理解：是\n是否答完整：否\n判定来源：fallback_heuristic\n判定理由：模型输出不是可解析的 JSON，已使用本地 fallback。\n教学模式：socratic_light\n本轮策略话术：学生方向基本正确，但当前题还没答完整。先肯定，再只追问当前题缺的那一步，不要给变式。\n\n【回复要求】\n这是第一轮 coach 回复，必

RuntimeError: Session already ended: student_understood

In [ ]:
print('=== Final Session Summary ===')
print(json.dumps({
    'turn_index': coach_session.turn_index,
    'done': coach_session.done,
    'stop_reason': coach_session.stop_reason,
    'messages_len': len(coach_session.messages),
}, ensure_ascii=False, indent=2))